# 手写数字识别的神经网络，二元分类

在本练习中，你将使用神经网络识别手写数字 0 和 1。


# 目录
- [ 1 - 程序包 ](#1)
- [ 2 - 神经网络](#2)
  - [ 2.1 问题陈述](#2.1)
  - [ 2.2 数据集](#2.2)
  - [ 2.3 模型表示](#2.3)
  - [ 2.4 Tensorflow 模型实现](#2.4)
    - [ 练习 1](#ex01)
  - [ 2.5 NumPy 模型实现（NumPy 中的前向传播）](#2.5)
    - [ 练习 2](#ex02)
  - [ 2.6 向量化 NumPy 模型实现（可选）](#2.6)
    - [ 练习 3](#ex03)
  - [ 2.7 恭喜！](#2.7)
  - [ 2.8 NumPy 广播教程（可选）](#2.8)


<a name="1"></a>
## 1 - 程序包

首先，让我们运行下面的单元格，导入本作业需要用到的所有程序包。
- [numpy](https://numpy.org/) 是 Python 科学计算的基础程序包。
- [matplotlib](http://matplotlib.org) 是 Python 中常用的绘图库。
- [tensorflow](https://www.tensorflow.org/) 是流行的机器学习平台。

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
from autils import *
%matplotlib inline

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

**Tensorflow 与 Keras**  
Tensorflow 是 Google 开发的机器学习程序包。2019 年，Google 把 Keras 集成进 Tensorflow，并发布了 Tensorflow 2.0。Keras 是 François Chollet 独立开发的框架，它为 Tensorflow 提供了一个简单的、以层为中心的接口。本课程将使用 Keras 接口。

<a name="2"></a>
## 2 - 神经网络

在课程 1 中，你实现了逻辑回归。通过多项式回归，它被扩展以处理非线性边界。对于图像识别这类更复杂的场景，神经网络是更优选择。

<a name="2.1"></a>
### 2.1 问题陈述

在本练习中，你将使用神经网络识别两个手写数字：0 和 1。这是一个二元分类任务。如今，自动手写数字识别被广泛使用——从识别信封上的邮政编码到识别银行支票上写的金额。在将来的作业中，你将扩展这个网络以识别全部 10 个数字（0-9）。

本练习将向你展示如何把你学到的方法用于这个分类任务。

<a name="2.2"></a>
### 2.2 数据集

你将先加载本任务的数据集。
- 下面所示的 `load_data()` 函数把数据加载到变量 `X` 和 `y` 中


- 数据集包含 1000 个手写数字的训练样本$^1$，这里仅限 0 和 1。

    - 每个训练样本是一张 20 像素 × 20 像素的数字灰度图。
        - 每个像素用一个浮点数表示，代表该位置的灰度强度。
        - 20 × 20 的像素网格被“展开”成一个 400 维向量。
        - 每个训练样本成为数据矩阵 `X` 中的一行。
        - 这样我们就得到一个 1000 × 400 的矩阵 `X`，其中每一行是一张手写数字图像的训练样本。

$$X = 
\left(\begin{array}{cc} 
--- (x^{(1)}) --- \\
--- (x^{(2)}) --- \\
\vdots \\ 
--- (x^{(m)}) --- 
\end{array}\right)$$ 

- 训练集的第二部分是一个 1000 × 1 维向量 `y`，包含训练集的标签
    - 如果图像是数字 `0`，则 `y = 0`；如果图像是数字 `1`，则 `y = 1`。

$^1$<sub> 这是 MNIST 手写数字数据集的一个子集（http://yann.lecun.com/exdb/mnist/）</sub>

In [ ]:
# 加载数据集
X, y = load_data()

<a name="toc_89367_2.2.1"></a>
#### 2.2.1 查看变量
让我们更熟悉一下你的数据集。  
- 一个好的起点是把每个变量打印出来，看看它包含什么。

下面的代码打印变量 `X` 和 `y` 的元素。  

In [ ]:
print ('The first element of X is: ', X[0])

In [ ]:
print ('The first element of y is: ', y[0,0])
print ('The last element of y is: ', y[-1,0])

<a name="toc_89367_2.2.2"></a>
#### 2.2.2 检查变量的维度

熟悉数据的另一种方式是查看它的维度。请打印 `X` 和 `y` 的形状，看看数据集中有多少训练样本。

In [ ]:
print ('The shape of X is: ' + str(X.shape))
print ('The shape of y is: ' + str(y.shape))

<a name="toc_89367_2.2.3"></a>
#### 2.2.3 可视化数据

你将先可视化训练集的一个子集。
- 在下面的单元格中，代码从 `X` 中随机选取 64 行，把每一行映射回 20 像素 × 20 像素的灰度图，并把图像一起显示出来。
- 每张图像的标签显示在图像上方

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# 你不需要修改本单元格中的任何内容

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(8,8))
fig.tight_layout(pad=0.1)

for i,ax in enumerate(axes.flat):
    # 随机选取索引
    random_index = np.random.randint(m)
    
    # 选取与随机索引对应的行，并
    # 重塑图像
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # 显示图像
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # 在图像上方显示标签
    ax.set_title(y[random_index,0])
    ax.set_axis_off()

<a name="2.3"></a>
### 2.3 模型表示

本作业中你将使用的神经网络如下图所示。
- 它包含三个带 sigmoid 激活函数的密集层。
    - 回忆一下，我们的输入是数字图像的像素值。
    - 由于图像尺寸为 $20\times20$，因此我们有 $400$ 个输入  
    
<img src="图片/C2_W1_Assign1.PNG" width="500" height="400">

> **图中文字（中文）**：神经网络结构：输入（400 个特征）→ 25 个单元 → 15 个单元 → 1 个单元（输出）

- 这些参数的维度按这样一个神经网络来确定：第 1 层 $25$ 个单元，第 2 层 $15$ 个单元，第 3 层 $1$ 个输出单元。

    - 回忆一下，这些参数的维度按如下方式确定：
        - 如果网络某一层有 $s_{in}$ 个单元、下一层有 $s_{out}$ 个单元，那么
            - $W$ 的维度为 $s_{in} \\times s_{out}$。
            - $b$ 是一个含 $s_{out}$ 个元素的向量
  
    - 因此，`W` 和 `b` 的形状分别为
        - 第 1 层：`W1` 的形状是 (400, 25)，`b1` 的形状是 (25,)
        - 第 2 层：`W2` 的形状是 (25, 15)，`b2` 的形状是 (15,)
        - 第 3 层：`W3` 的形状是 (15, 1)，`b3` 的形状是 (1,)
>**注意：** 偏置向量 `b` 可以表示为 1 维数组 (n,) 或 2 维数组 (n,1)。Tensorflow 采用 1 维表示，本实验将沿用这一约定。
               

<a name="2.4"></a>
### 2.4 Tensorflow 模型实现


Tensorflow 模型是逐层构建的。层的输入维度（上面的 $s_{in}$）会自动为你计算。你指定层的*输出维度*，它决定了下一层的输入维度。第一层的输入维度来自下面 `model.fit` 语句中指定的输入数据大小。
>**注意：** 也可以添加一个输入层来指定第一层的输入维度。例如：  
`tf.keras.Input(shape=(400,)),    #specify input shape`  
我们这里会加上它，以说明模型的尺寸确定过程。

<a name="ex01"></a>
### 练习 1

下面，使用 Keras 的 [Sequential 模型](https://keras.io/guides/sequential_model/)和带 sigmoid 激活函数的[密集层](https://keras.io/api/layers/core_layers/dense/)来构建上面描述的网络。

In [ ]:
# UNQ_C1
# 计分单元：Sequential 模型

model = Sequential(
    [               
        tf.keras.Input(shape=(400,)),    #指定输入大小
        ### 在此处编写代码 ###
        
        ### 代码编写结束 ###
    ], name = "my_model" 
)                            


In [ ]:
model.summary()

<details>
  <summary><font size="3" color="darkgreen"><b>预期输出（点击展开）</b></font></summary>
<code>model.summary()</code> 函数会显示模型的实用摘要。由于我们指定了输入层尺寸，权重和偏置数组的形状就被确定下来，每层的参数总数也能显示出来。注意，层的名称是自动生成的，可能各不相同。  
    
    

```
Model: "my_model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
dense (Dense)                (None, 25)                10025     
_________________________________________________________________
dense_1 (Dense)              (None, 15)                390       
_________________________________________________________________
dense_2 (Dense)              (None, 1)                 16        
=================================================================
Total params: 10,431
Trainable params: 10,431
Non-trainable params: 0
_________________________________________________________________
```


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
正如课上所述：
    

```python
model = Sequential(                      
    [                                   
        tf.keras.Input(shape=(400,)),    # specify input size (optional)
        Dense(25, activation='sigmoid'), 
        Dense(15, activation='sigmoid'), 
        Dense(1,  activation='sigmoid')  
    ], name = "my_model"                                    
)                                       
```
 

In [ ]:
# 单元测试
from public_tests import * 

test_c1(model)

摘要中显示的参数数量对应下面所示的权重和偏置数组中的元素个数。

In [ ]:
L1_num_params = 400 * 25 + 25  # W1 参数 + b1 参数
L2_num_params = 25 * 15 + 15   # W2 参数 + b2 参数
L3_num_params = 15 * 1 + 1     # W3 参数 + b3 参数
print("L1 params = ", L1_num_params, ", L2 params = ", L2_num_params, ",  L3 params = ", L3_num_params )

让我们进一步检查权重，验证 tensorflow 生成的维度与我们上面计算的一致。

In [ ]:
[layer1, layer2, layer3] = model.layers

In [ ]:
#### 检查权重的形状
W1,b1 = layer1.get_weights()
W2,b2 = layer2.get_weights()
W3,b3 = layer3.get_weights()
print(f"W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"W2 shape = {W2.shape}, b2 shape = {b2.shape}")
print(f"W3 shape = {W3.shape}, b3 shape = {b3.shape}")

**预期输出**

```
W1 shape = (400, 25), b1 shape = (25,)  
W2 shape = (25, 15), b2 shape = (15,)  
W3 shape = (15, 1), b3 shape = (1,)
```


`xx.get_weights` 返回一个 NumPy 数组。也可以直接以张量形式访问权重。注意最后一层张量的形状。

In [ ]:
print(model.layers[2].weights)

下面的代码将定义损失函数并运行梯度下降，把模型的权重拟合到训练数据上。这将在下周详细解释。

In [ ]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X,y,
    epochs=20
)

要在某个样本上运行模型做预测，请使用 [Keras `predict`](https://www.tensorflow.org/api_docs/python/tf/keras/Model)。`predict` 的输入是一个数组，所以单个样本要 reshape 成二维。

In [ ]:
prediction = model.predict(X[0].reshape(1,400))  # 一个 0
print(f" predicting a zero: {prediction}")
prediction = model.predict(X[500].reshape(1,400))  # 一个 1
print(f" predicting a one:  {prediction}")

模型的输出被解释为一个概率。在上面的第一个例子中，输入是 0。模型预测输入为 1 的概率几乎为零。  
在第二个例子中，输入是 1。模型预测输入为 1 的概率接近于一。  
与逻辑回归的情形一样，把该概率与一个阈值比较，以得到最终预测。

In [ ]:
if prediction >= 0.5:
    yhat = 1
else:
    yhat = 0
print(f"prediction after threshold: {yhat}")

让我们把随机抽取的 64 个数字的预测结果与标签做比较。这需要运行一会儿。

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# 你不需要修改本单元格中的任何内容

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(8,8))
fig.tight_layout(pad=0.1,rect=[0, 0.03, 1, 0.92]) #[左, 下, 右, 上]

for i,ax in enumerate(axes.flat):
    # 随机选取索引
    random_index = np.random.randint(m)
    
    # 选取与随机索引对应的行，并
    # 重塑图像
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # 显示图像
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # 使用神经网络做预测
    prediction = model.predict(X[random_index].reshape(1,400))
    if prediction >= 0.5:
        yhat = 1
    else:
        yhat = 0
    
    # 在图像上方显示标签
    ax.set_title(f"{y[random_index,0]},{yhat}")
    ax.set_axis_off()
fig.suptitle("Label, yhat", fontsize=16)
plt.show()

<a name="2.5"></a>
### 2.5 NumPy 模型实现（NumPy 中的前向传播）
正如课上所述，你可以用 NumPy 构建自己的密集层。这样就可以用它来构建多层神经网络。

<img src="图片/C2_W1_dense2.PNG" width="600" height="450">

> **图中文字（中文）**：神经网络层 def my_dense(a_in, W, b, g):     units = W.shape[1]     a_out = np.zeros(units)     for j in range(units):         z = np.dot(W[:, j], a_in) + b[j]         a_out[j] = g(z)     return a_out


<a name="ex02"></a>
### 练习 2

下面，构建一个密集层子程序。课上的例子用一个 for 循环访问层中的每个单元（`j`），把该单元的权重（`W[:,j]`）与输入做点积，再加上该单元的偏置（`b[j]`），得到 `z`。然后对结果应用激活函数 `g(z)`。本节不会使用可选课程中介绍的某些矩阵运算。那些将在后面的小节中探讨。

In [ ]:
# UNQ_C2
# 计分函数：my_dense

def my_dense(a_in, W, b, g):
    """计算全连接层
    参数：
      a_in (ndarray (n, )) : 数据，1 个样本
      W    (ndarray (n,j)) : 权重矩阵，每个单元 n 个特征，共 j 个单元
      b    (ndarray (j, )) : 偏置向量，j 个单元
      g    激活函数（例如 sigmoid、relu……）
    返回
      a_out (ndarray (j,))  : j 个单元"""
    units = W.shape[1]
    a_out = np.zeros(units)
### 在此处编写代码 ###
        
### 代码编写结束 ###
    return(a_out)


In [ ]:
# 快速检查
x_tst = 0.1*np.arange(1,3,1).reshape(2,)  # （1 个样本，2 个特征）
W_tst = 0.1*np.arange(1,7,1).reshape(2,3) # （2 个输入特征，3 个输出特征）
b_tst = 0.1*np.arange(1,4,1).reshape(3,)  # （3 个特征）
A_tst = my_dense(x_tst, W_tst, b_tst, sigmoid)
print(A_tst)

**预期输出**

```
[0.54735762 0.57932425 0.61063923]
```


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
正如课上所述：
    

```python
def my_dense(a_in, W, b, g):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      a_out (ndarray (j,))  : j units
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):             
        w =                            # Select weights for unit j. These are in column j of W
        z =                            # dot product of w and a_in + b
        a_out[j] =                     # apply activation to z
    return(a_out)
```

   
    
<details>
  <summary><font size="3" color="darkgreen"><b>点击查看更多提示</b></font></summary>

    

```python
def my_dense(a_in, W, b, g):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      a_out (ndarray (j,))  : j units
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):             
        w = W[:,j]                     
        z = np.dot(w, a_in) + b[j]     
        a_out[j] = g(z)                
    return(a_out)
```
 

In [ ]:
# 单元测试
test_c2(my_dense)

下面的单元格利用上面的 `my_dense` 子程序构建了一个三层神经网络。

In [ ]:
def my_sequential(x, W1, b1, W2, b2, W3, b3):
    a1 = my_dense(x,  W1, b1, sigmoid)
    a2 = my_dense(a1, W2, b2, sigmoid)
    a3 = my_dense(a2, W3, b3, sigmoid)
    return(a3)

我们可以从 Tensorflow 复制已训练好的权重和偏置。

In [ ]:
W1_tmp,b1_tmp = layer1.get_weights()
W2_tmp,b2_tmp = layer2.get_weights()
W3_tmp,b3_tmp = layer3.get_weights()

In [ ]:
# 做预测
prediction = my_sequential(X[0], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
if prediction >= 0.5:
    yhat = 1
else:
    yhat = 0
print( "yhat = ", yhat, " label= ", y[0,0])
prediction = my_sequential(X[500], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
if prediction >= 0.5:
    yhat = 1
else:
    yhat = 0
print( "yhat = ", yhat, " label= ", y[500,0])

运行下面的单元格，查看 NumPy 模型和 Tensorflow 模型的预测结果。这需要运行一会儿。

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# 你不需要修改本单元格中的任何内容

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(8,8))
fig.tight_layout(pad=0.1,rect=[0, 0.03, 1, 0.92]) #[左, 下, 右, 上]

for i,ax in enumerate(axes.flat):
    # 随机选取索引
    random_index = np.random.randint(m)
    
    # 选取与随机索引对应的行，并
    # 重塑图像
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # 显示图像
    ax.imshow(X_random_reshaped, cmap='gray')

    # 用 NumPy 实现的神经网络做预测
    my_prediction = my_sequential(X[random_index], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
    my_yhat = int(my_prediction >= 0.5)

    # 用 Tensorflow 实现的神经网络做预测
    tf_prediction = model.predict(X[random_index].reshape(1,400))
    tf_yhat = int(tf_prediction >= 0.5)
    
    # 在图像上方显示标签
    ax.set_title(f"{y[random_index,0]},{tf_yhat},{my_yhat}")
    ax.set_axis_off() 
fig.suptitle("Label, yhat Tensorflow, yhat Numpy", fontsize=16)
plt.show()

<a name="2.6"></a>
### 2.6 向量化 NumPy 模型实现（可选）
可选课程介绍了可用于加速计算的向量和矩阵运算。
下面描述的是给定输入样本时计算一层中所有单元输出的层运算：

<img src="图片/C2_W1_VectorMatrix.PNG" width="600" height="450">

> **图中文字（中文）**：Z = XW（向量乘矩阵） 维度必须匹配

我们可以用上面的样本 `X` 和参数 `W1`、`b1` 来演示这一点。我们用 `np.matmul` 执行矩阵乘法。注意，x 和 W 的维度必须像上图那样相互匹配。

In [ ]:
x = X[0].reshape(-1,1)         # 列向量 (400,1)
z1 = np.matmul(x.T,W1) + b1    # (1,400)(400,25) = (1,25)
a1 = sigmoid(z1)
print(a1.shape)

你可以更进一步，用一次矩阵-矩阵运算计算所有样本的所有单元。

<img src="图片/C2_W1_MatrixMatrix.PNG" width="600" height="450">

> **图中文字（中文）**：Z = XW（矩阵乘矩阵） 维度：X 的列数必须与 W 的行数匹配
完整运算是 $\\mathbf{Z}=\\mathbf{XW}+\\mathbf{b}$。这里会利用 NumPy 广播把 $\\mathbf{b}$ 扩展为 $m$ 行。如果对此不熟悉，notebook 末尾提供了一个简短教程。

<a name="ex03"></a>
### 练习 3

下面，编写一个新的 `my_dense_v` 子程序，对样本矩阵执行层计算。这里会用到 `np.matmul()`。

In [ ]:
# UNQ_C3
# 计分函数：my_dense_v

def my_dense_v(A_in, W, b, g):
    """计算全连接层
    参数：
      A_in (ndarray (m,n)) : 数据，m 个样本，每个样本 n 个特征
      W    (ndarray (n,j)) : 权重矩阵，每个单元 n 个特征，共 j 个单元
      b    (ndarray (j,1)) : 偏置向量，j 个单元
      g    激活函数（例如 sigmoid、relu……）
    返回
      A_out (ndarray (m,j)) : m 个样本，j 个单元"""
### 在此处编写代码 ###
    
    
### 代码编写结束 ###
    return(A_out)

In [ ]:
X_tst = 0.1*np.arange(1,9,1).reshape(4,2) # （4 个样本，2 个特征）
W_tst = 0.1*np.arange(1,7,1).reshape(2,3) # （2 个输入特征，3 个输出特征）
b_tst = 0.1*np.arange(1,4,1).reshape(1,3) # （3 个特征，1）
A_tst = my_dense_v(X_tst, W_tst, b_tst, sigmoid)
print(A_tst)

**预期输出**


```
[[0.54735762 0.57932425 0.61063923]
 [0.57199613 0.61301418 0.65248946]
 [0.5962827  0.64565631 0.6921095 ]
 [0.62010643 0.67699586 0.72908792]]
 ```


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    写成矩阵形式，这可以用一到两行完成。
    
       Z = np.matmul of A_in and W plus b    
       A_out is g(Z)  
<details>
  <summary><font size="3" color="darkgreen"><b>点击查看代码</b></font></summary>


```python
def my_dense_v(A_in, W, b, g):
    """
    Computes dense layer
    Args:
      A_in (ndarray (m,n)) : Data, m examples, n features each
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j,1)) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      A_out (ndarray (m,j)) : m examples, j units
    """
    Z = np.matmul(A_in,W) + b    
    A_out = g(Z)                 
    return(A_out)
```



In [ ]:
# 单元测试
test_c3(my_dense_v)

下面的单元格利用上面的 `my_dense_v` 子程序构建了一个三层神经网络。

In [ ]:
def my_sequential_v(X, W1, b1, W2, b2, W3, b3):
    A1 = my_dense_v(X,  W1, b1, sigmoid)
    A2 = my_dense_v(A1, W2, b2, sigmoid)
    A3 = my_dense_v(A2, W3, b3, sigmoid)
    return(A3)

我们同样可以从 Tensorflow 复制已训练好的权重和偏置。

In [ ]:
W1_tmp,b1_tmp = layer1.get_weights()
W2_tmp,b2_tmp = layer2.get_weights()
W3_tmp,b3_tmp = layer3.get_weights()

让我们用新模型做一次预测。这将*一次性对所有样本*做出预测。注意输出的形状。

In [ ]:
Prediction = my_sequential_v(X, W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
Prediction.shape

和之前一样，我们将应用 0.5 的阈值，但这次是对所有预测一次性应用。

In [ ]:
Yhat = (Prediction >= 0.5).numpy().astype(int)
print("predict a zero: ",Yhat[0], "predict a one: ", Yhat[500])

运行下面的单元格查看预测结果。它会使用我们上面刚算出的预测。这需要运行一会儿。

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# 你不需要修改本单元格中的任何内容

m, n = X.shape

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1, rect=[0, 0.03, 1, 0.92]) #[左, 下, 右, 上]

for i, ax in enumerate(axes.flat):
    # 随机选取索引
    random_index = np.random.randint(m)
    
    # 选取与随机索引对应的行，并
    # 重塑图像
    X_random_reshaped = X[random_index].reshape((20, 20)).T
    
    # 显示图像
    ax.imshow(X_random_reshaped, cmap='gray')
   
    # 在图像上方显示标签
    ax.set_title(f"{y[random_index,0]}, {Yhat[random_index, 0]}")
    ax.set_axis_off() 
fig.suptitle("Label, Yhat", fontsize=16)
plt.show()

你可以看看其中一张被错误分类的图像长什么样。

In [ ]:
fig = plt.figure(figsize=(1, 1))
errors = np.where(y != Yhat)
random_index = errors[0][0]
X_random_reshaped = X[random_index].reshape((20, 20)).T
plt.imshow(X_random_reshaped, cmap='gray')
plt.title(f"{y[random_index,0]}, {Yhat[random_index, 0]}")
plt.axis('off')
plt.show()

<a name="2.7"></a>
### 2.7 恭喜！
你已经成功构建并使用了神经网络。

<a name="2.8"></a>
### 2.8 NumPy 广播教程（可选）


在最后一个例子中，$\\mathbf{Z}=\\mathbf{XW} + \\mathbf{b}$ 利用 NumPy 广播扩展了向量 $\\mathbf{b}$。如果你不熟悉 NumPy 广播，这里提供一个简短教程。

$\\mathbf{XW}$ 是一个矩阵-矩阵运算，维度为 $(m,j_1)(j_1,j_2)$，结果为维度 $(m,j_2)$ 的矩阵。在此基础上，我们加上维度为 $(j_2,)$ 的向量 $\\mathbf{b}$。要使这个逐元素运算成立，$\\mathbf{b}$ 必须被扩展为 $(m,j_2)$ 矩阵。这个扩展由 NumPy 广播为你完成。

广播适用于逐元素运算。  
它的基本操作是通过复制元素把较小的维度“拉伸”到与较大维度相匹配。

更[具体地说](https://NumPy.org/doc/stable/user/basics.broadcasting.html)：
对两个数组做运算时，NumPy 会逐元素比较它们的形状。它从末尾（即最右侧）的维度开始，向左逐维推进。两个维度在满足以下条件时是兼容的
- 它们相等，或者
- 其中之一为 1   

如果这些条件不满足，就会抛出 ValueError: operands could not be broadcast together 异常，表明数组的形状不兼容。结果数组的尺寸，就是各输入轴向上不为 1 的那个尺寸。

下面是一些例子：

<figure>
    <center> <img src="./图片/C2_W1_Assign1_BroadcastIndexes.PNG"  alt='missing' width="400"  >

> **图中文字（中文）**：广播索引示意：m×n 矩阵与向量运算，结果形状为 m×n<center/>
    <figcaption>计算广播结果的形状</figcaption>
<figure/>

下面的图描述了维度扩展。注意下面的红色文字：

<figure>
    <center> <img src="./图片/C2_W1_Assign1_Broadcasting.gif"  alt='missing' width="600"  >

> **图中文字（中文）**：NumPy 广播：向量与标量（a：4×1）<center/>
    <figcaption>广播在概念上扩展参数，使逐元素运算能够匹配</figcaption>
<figure/>

上图展示了 NumPy 在执行最终运算之前扩展参数以使其匹配。注意，这是一种概念性的描述。NumPy 运算的实际机制会选择最高效的实现。

对于下面的每个例子，请在运行之前先试着猜一下结果的大小。

In [ ]:
a = np.array([1,2,3]).reshape(-1,1)  #(3,1)
b = 5
print(f"(a + b).shape: {(a + b).shape}, \na + b = \n{a + b}")

注意，这适用于所有逐元素运算：

In [ ]:
a = np.array([1,2,3]).reshape(-1,1)  #(3,1)
b = 5
print(f"(a * b).shape: {(a * b).shape}, \na * b = \n{a * b}")

<figure>
    <img src="./图片/C2_W1_Assign1_VectorAdd.PNG"  alt='missing' width="740" >

> **图中文字（中文）**：向量加法：a + b = b + a [0, 0, 0, 0] + b = [b1, b2, …] a0 + 0、a1 + 0、a2 + 0、a3 + 0 a0 + 1、a1 + 1、a2 + 1、a3 + 1 a0 + 2、a1 + 2、a2 + 2、a3 + 2
    <center><figcaption><b>行-列逐元素运算</b></figcaption></center>
<figure/>

In [ ]:
a = np.array([1,2,3,4]).reshape(-1,1)
b = np.array([1,2,3]).reshape(1,-1)
print(a)
print(b)
print(f"(a + b).shape: {(a + b).shape}, \na + b = \n{a + b}")

这就是你上面构建的密集层中的场景：把一个 1 维向量 $b$ 加到一个 (m,j) 矩阵上。
<figure>
    <img src="./图片/C2_W1_Assign1_BroadcastMatrix.PNG"  alt='missing' width="740" >

> **图中文字（中文）**：广播示意：a（4×3）与 b（3，一维向量）运算得到结果 r（4×3） B 是一个一维向量
    <center><figcaption><b>矩阵 + 1 维向量</b></figcaption></center>
<figure/>